# Experimento: ¿podemos mejorar el modelo con LightGBM (y alternativas)?

Cuaderno **separado** del análisis principal (`analisis_completo.ipynb`). No modifica el pipeline Kedro: si algo aquí resulta mejor *de forma defendible*, se adopta después.

**Reglas del experimento (para no engañarnos):**
1. Todos los modelos se comparan con **los mismos 5 pliegues agrupados** (`GroupKFold` por la llave "gemelos"), sobre el conjunto de entrenamiento.
2. El **conjunto de prueba se usa una sola vez**, al final, con los modelos ya fijados.
3. Solo variables conocidas antes de la compra (edad, género, país, estado, mes). Nada de producto, precio, cantidad ni año.
4. Una mejora cuenta solo si supera el ruido entre pliegues (±0.003 de AUC aprox.).
5. Todo número en este cuaderno sale de una celda ejecutada.

Referencias: LightGBM [Ke et al., 2017], CatBoost [Prokhorenkova et al., 2018], Optuna [Akiba et al., 2019]; listadas al final.

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
from bike_propensity.pipelines.data_processing.nodes import build_model_input, clean_sales  # noqa: E402
from bike_propensity.pipelines.model_training.nodes import make_hgb_pipeline  # noqa: E402

DATA, FIG = ROOT / "data", ROOT / "docs" / "figures"
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight"})
BLUE, ORANGE, GRAY = "#2a6f97", "#e07a1f", "#8d99ae"
SEED = 42

mi = pd.read_parquet(DATA / "03_primary" / "model_input.parquet")  # ya con la ventana estable del pipeline
train_full = pd.read_parquet(DATA / "05_model_input" / "train.parquet")
test_full = pd.read_parquet(DATA / "05_model_input" / "test.parquet")
prod = pd.read_pickle(DATA / "06_models" / "final_hgb.pkl")
print(train_full.shape, test_full.shape)

## 1. Hallazgo previo: el catálogo cambia durante el periodo (auditoría de datos)

Antes de comparar algoritmos revisé *qué población* estamos modelando. En el dataset crudo, **Accesorios y Ropa no existen antes de julio de 2013** (antes solo se vendían bicicletas), y en **julio de 2014 no hay ninguna bicicleta**. Esos meses son regímenes distintos, no comportamiento del cliente.

In [ ]:
raw = pd.read_csv(DATA / "01_raw" / "sales_data.csv", parse_dates=["Date"]).drop_duplicates()
raw["bike"] = raw.Product_Category == "Bikes"
audit = raw.groupby(raw.Date.dt.to_period("M")).agg(pedidos=("bike", "size"), pct_bicicleta=("bike", "mean"), categorias=("Product_Category", "nunique"))
audit["pct_bicicleta"] = (audit.pct_bicicleta * 100).round(1)
print("Primera venta por categoría:"); print(raw.groupby("Product_Category").Date.min().dt.date.to_string())
audit.loc["2013-04":"2014-08"]

**Decisión (ya incorporada al pipeline):** `clean_sales` conserva un mes solo si tiene las 3 categorías **y** el mes anterior también las tenía. Eso deja ago-2013→jun-2014 y, por la estructura de gemelos, ago-2015→jun-2016 (22 meses). Aquí medimos si esa decisión cambia el resultado, comparando con la ventana anterior (todo desde 2013).

In [ ]:
tr, te = train_full.reset_index(drop=True), test_full.reset_index(drop=True)   # el pipeline ya aplica la ventana estable
print(f"Ventana estable (pipeline): train {len(tr):,} | test {len(te):,} | prevalencia train {tr.is_bike.mean():.3f} | test {te.is_bike.mean():.3f}")
print("Grupos train/test solapados:", len(set(tr.split_group) & set(te.split_group)), "(debe ser 0)")

# Ventana anterior (todo desde 2013, sin la regla de catálogo), reconstruida con las mismas funciones de limpieza
raw_csv = pd.read_csv(DATA / "01_raw" / "sales_data.csv")
clean_old, _ = clean_sales(raw_csv, False)
clean_old = clean_old[clean_old.Date >= "2013-01-01"].reset_index(drop=True)
GROUP_COLS = ["Day", "Month", "Customer_Age", "Customer_Gender", "State"]
mi_old = build_model_input(clean_old, "Bikes", GROUP_COLS)
def gkey(df): return df.day_of_month.astype(str) + "|" + df.month.astype(str) + "|" + df.Customer_Age.astype(str) + "|" + df.Customer_Gender + "|" + df.State
print(f"Ventana anterior (>=2013): {len(mi_old):,} filas | prevalencia {mi_old.is_bike.mean():.3f}")

## 2. Preparación común y validación

Mismos pliegues para todos los modelos. Las categóricas se declaran con el **mismo conjunto de niveles** en entrenamiento y prueba.

In [ ]:
CATS, NUMS = ["Customer_Gender", "Country", "State"], ["Customer_Age", "month"]
LEVELS = {c: sorted(mi[c].unique()) for c in CATS}

def feats(df, cat_dtype=True):
    X = df[CATS + NUMS].copy()
    for c in CATS:
        X[c] = pd.Categorical(X[c], categories=LEVELS[c]) if cat_dtype else X[c].astype(str)
    return X

X, y, g = feats(tr), tr.is_bike.values, tr.split_group.values
FOLDS = list(GroupKFold(5).split(X, y, g))
RES = {}   # nombre -> dict(oof, auc_folds, ap_folds)

def cv_eval(name, fit_predict, verbose=True):
    t0 = time.time(); oof = np.zeros(len(y)); aucs, aps = [], []
    for tri, vai in FOLDS:
        p = fit_predict(tri, vai); oof[vai] = p
        aucs.append(roc_auc_score(y[vai], p)); aps.append(average_precision_score(y[vai], p))
    RES[name] = dict(oof=oof, auc=np.array(aucs), ap=np.array(aps), secs=time.time() - t0)
    if verbose:
        print(f"{name:42s} AUC {np.mean(aucs):.4f} ± {np.std(aucs):.4f} | AP {np.mean(aps):.4f} | {time.time()-t0:5.0f}s")

# Referencia: el modelo actual (HGB) con los hiperparámetros ya elegidos en el pipeline principal
HGB_KEYS = ["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization", "max_depth"]
hgb_params = {k: v for k, v in prod.named_steps["model"].get_params().items() if k in HGB_KEYS}
def fit_hgb(tri, vai):
    m = make_hgb_pipeline(CATS, NUMS, SEED).set_params(**{f"model__{k}": v for k, v in hgb_params.items()})
    m.fit(tr.iloc[tri][CATS + NUMS], y[tri]); return m.predict_proba(tr.iloc[vai][CATS + NUMS])[:, 1]
cv_eval("HGB (modelo actual)", fit_hgb)

### 2.1 ¿Depende el resultado de la ventana? (modelo actual en ambas)

In [ ]:
def cv_on(df, cols_cats, cols_nums):
    yy = df.is_bike.values; a, p = [], []
    for tri, vai in GroupKFold(5).split(df, yy, df.split_group):
        m = make_hgb_pipeline(cols_cats, cols_nums, SEED).set_params(**{f"model__{k}": v for k, v in hgb_params.items()})
        m.fit(df.iloc[tri][cols_cats + cols_nums], yy[tri]); pr = m.predict_proba(df.iloc[vai][cols_cats + cols_nums])[:, 1]
        a.append(roc_auc_score(yy[vai], pr)); p.append(average_precision_score(yy[vai], pr))
    return np.mean(a), np.std(a), np.mean(p), yy.mean()
mi_new = pd.concat([tr, te], ignore_index=True)
rows = []
for nm, df in [("Ventana anterior (≥2013)", mi_old), ("Ventana estable (pipeline)", mi_new)]:
    for vn, nums in [("con mes", NUMS), ("sin mes", ["Customer_Age"])]:
        a, s_, p, prev = cv_on(df, CATS, nums); rows.append(dict(ventana=nm, variables=vn, AUC=a, sd=s_, AP=p, prevalencia=prev, lift_AP=p / prev))
pd.DataFrame(rows).round(4)

INTERP_WINDOW

## 3. Candidatos

### 3.1 Techo informativo: ¿cuánta señal hay realmente?
Una tabla de consulta suavizada por celdas (estado × tramo de edad × mes) es un "modelo" sin aprendizaje. Si los modelos complejos no la superan por mucho, la señal está agotada.

In [ ]:
trk = tr.assign(age_band=pd.cut(tr.Customer_Age, [16, 22, 26, 30, 34, 38, 42, 46, 50, 54, 58, 62, 100]).astype(str))
def lookup(keys, k=20):
    def f(tri, vai):
        d = trk.iloc[tri]; prior = d.is_bike.mean()
        s = d.groupby(keys).is_bike.agg(["sum", "count"]); r = ((s["sum"] + k * prior) / (s["count"] + k)).rename("r")
        return trk.iloc[vai][keys].merge(r, left_on=keys, right_index=True, how="left")["r"].fillna(prior).values
    return f
cv_eval("Tabla de consulta estado×edad×mes", lookup(["State", "age_band", "month"]))
cv_eval("Tabla de consulta país×edad×mes", lookup(["Country", "age_band", "month"]))

### 3.2 LightGBM con categóricas nativas (valores por defecto)
LightGBM usa histogramas y crecimiento por hojas (*leaf-wise*) [Ke et al., 2017]; maneja categóricas dividiéndolas por el orden de su gradiente (Fisher, 1958, vía la implementación de LightGBM).

In [ ]:
def fit_lgb(params):
    def f(tri, vai):
        m = lgb.LGBMClassifier(verbose=-1, random_state=SEED, n_jobs=4, **params)
        m.fit(X.iloc[tri], y[tri]); return m.predict_proba(X.iloc[vai])[:, 1]
    return f
cv_eval("LightGBM (valores por defecto)", fit_lgb({}))

### 3.3 LightGBM ajustado con Optuna (60 intentos)
Búsqueda bayesiana (TPE) [Akiba et al., 2019] optimizando PR-AUC medio en los 5 pliegues. Espacio amplio con regularización fuerte para categóricas de cardinalidad media (`cat_smooth`, `cat_l2`, `min_data_per_group`) y `path_smooth`.

*Cautela:* con 60 intentos, el mejor puntaje de CV está ligeramente sesgado al alza (selección sobre el mismo CV). Por eso la decisión final usa el conjunto de prueba intacto.

In [ ]:
def objective(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000, step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        num_leaves=trial.suggest_int("num_leaves", 4, 64),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 600),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 50, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        cat_smooth=trial.suggest_float("cat_smooth", 1, 100, log=True),
        cat_l2=trial.suggest_float("cat_l2", 1, 50, log=True),
        min_data_per_group=trial.suggest_int("min_data_per_group", 20, 400),
        path_smooth=trial.suggest_float("path_smooth", 0, 50),
    )
    aps = []
    for tri, vai in FOLDS:
        m = lgb.LGBMClassifier(verbose=-1, random_state=SEED, n_jobs=4, **p).fit(X.iloc[tri], y[tri])
        aps.append(average_precision_score(y[vai], m.predict_proba(X.iloc[vai])[:, 1]))
    return float(np.mean(aps))

t0 = time.time()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=60)
print(f"Optuna: {time.time()-t0:.0f}s | mejor AP-CV = {study.best_value:.4f}")
BEST_LGB = study.best_params | {"subsample_freq": 1}
print(json.dumps(BEST_LGB, indent=1))
cv_eval("LightGBM (Optuna, 60 intentos)", fit_lgb(BEST_LGB))

In [ ]:
vals = [t.value for t in study.trials]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(vals, "o", color=GRAY, ms=4); ax[0].plot(np.maximum.accumulate(vals), color=BLUE, lw=2)
ax[0].set(xlabel="Intento", ylabel="PR-AUC medio (CV)", title="Búsqueda Optuna: mejor acumulado")
ax[1].hist(vals, bins=15, color=BLUE); ax[1].set(xlabel="PR-AUC medio (CV)", title="Distribución de los 60 intentos")
fig.tight_layout(); fig.savefig(FIG / "lgbm_optuna.png"); plt.show()
print(f"Rango de PR-AUC entre intentos: {min(vals):.4f} – {max(vals):.4f}")

### 3.4 CatBoost
Maneja categóricas con *ordered target statistics*, diseñadas para evitar fuga del objetivo [Prokhorenkova et al., 2018]; el candidato más plausible para `State`.

In [ ]:
Xcb = feats(tr, cat_dtype=False)
def fit_cb(tri, vai):
    m = CatBoostClassifier(iterations=400, learning_rate=0.07, depth=6, l2_leaf_reg=5, verbose=0, random_seed=SEED, thread_count=4, cat_features=CATS)
    m.fit(Xcb.iloc[tri], y[tri]); return m.predict_proba(Xcb.iloc[vai])[:, 1]
cv_eval("CatBoost (depth 6, 400 it.)", fit_cb)

### 3.5 GAM logístico (splines suaves) — alternativa interpretable
Regresión logística con *spline* cúbico en la edad, one-hot de estado, mes y género, y la interacción edad × país. Es un modelo transparente: si compite con el boosting, tendría un argumento fuerte de interpretabilidad (criterio de la guía).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
def make_gam(C=0.3):
    pre = ColumnTransformer([
        ("age", SplineTransformer(n_knots=8, degree=3, include_bias=False), ["Customer_Age"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), ["State", "month", "Customer_Gender"]),
    ])
    return Pipeline([("pre", pre), ("lr", LogisticRegression(C=C, max_iter=2000))])
def fit_gam(C):
    def f(tri, vai):
        m = make_gam(C).fit(tr.iloc[tri][CATS + NUMS].assign(month=lambda d: d.month.astype(str)), y[tri])
        return m.predict_proba(tr.iloc[vai][CATS + NUMS].assign(month=lambda d: d.month.astype(str)))[:, 1]
    return f
for C in (0.05, 0.3, 1.0):
    cv_eval(f"GAM logístico (C={C})", fit_gam(C))

In [ ]:
# GAM con interacción edad × país (splines de edad por país)
from sklearn.base import BaseEstimator, TransformerMixin
class AgeByCountry(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.sp_ = SplineTransformer(n_knots=6, degree=3, include_bias=False).fit(X[["Customer_Age"]])
        self.countries_ = sorted(X.Country.unique()); return self
    def transform(self, X):
        S = self.sp_.transform(X[["Customer_Age"]]); out = []
        for c in self.countries_:
            out.append(S * (X.Country.values == c)[:, None])
        return np.hstack(out)
def make_gam2(C=0.3):
    pre = ColumnTransformer([
        ("agec", AgeByCountry(), ["Customer_Age", "Country"]),
        ("age", SplineTransformer(n_knots=8, degree=3, include_bias=False), ["Customer_Age"]),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), ["State", "month", "Customer_Gender"]),
    ])
    return Pipeline([("pre", pre), ("lr", LogisticRegression(C=C, max_iter=3000))])
def fit_gam2(C):
    def f(tri, vai):
        a = tr.iloc[tri][CATS + NUMS].assign(month=lambda d: d.month.astype(str)); b = tr.iloc[vai][CATS + NUMS].assign(month=lambda d: d.month.astype(str))
        return make_gam2(C).fit(a, y[tri]).predict_proba(b)[:, 1]
    return f
for C in (0.1, 0.3):
    cv_eval(f"GAM logístico + edad×país (C={C})", fit_gam2(C))

### 3.6 Ingeniería de variables legítima (sobre LightGBM ajustado)
Ideas *pre-compra* que no introducen fuga: (a) el tramo de edad oficial del dataset (`Age_Group`: <25, 25–34, 35–64, 64+); (b) mes como categoría; (c) sin género; (d) edad × país como categoría combinada (frecuencia mínima).

In [ ]:
def build_variant(df, variant):
    d = feats(df);
    if variant == "base": return d
    if variant == "+Age_Group":
        d["age_group"] = pd.Categorical(pd.cut(df.Customer_Age, [0, 24, 34, 64, 200], labels=["<25", "25-34", "35-64", "64+"])); return d
    if variant == "mes categórico":
        d["month"] = pd.Categorical(df.month.astype(str)); return d
    if variant == "sin género": return d.drop(columns=["Customer_Gender"])
    if variant == "+país×tramo10años":
        d["ctry_age"] = pd.Categorical(df.Country.astype(str) + "_" + (df.Customer_Age // 10).astype(str)); return d
def fit_lgb_var(variant):
    Xv = build_variant(tr, variant)
    def f(tri, vai):
        m = lgb.LGBMClassifier(verbose=-1, random_state=SEED, n_jobs=4, **BEST_LGB).fit(Xv.iloc[tri], y[tri])
        return m.predict_proba(Xv.iloc[vai])[:, 1]
    return f
for v in ["+Age_Group", "mes categórico", "sin género", "+país×tramo10años"]:
    cv_eval(f"LGBM + variante: {v}", fit_lgb_var(v))

### 3.7 Mezcla (ensamble) de modelos
Promedio de rangos de las predicciones fuera de muestra (OOF) con **pesos iguales fijados de antemano** (sin optimizar pesos, para no sobreajustar el CV).

In [ ]:
def rank01(p): return rankdata(p) / len(p)
BLENDS = {
    "Mezcla HGB+LGBM": ["HGB (modelo actual)", "LightGBM (Optuna, 60 intentos)"],
    "Mezcla HGB+LGBM+CatBoost": ["HGB (modelo actual)", "LightGBM (Optuna, 60 intentos)", "CatBoost (depth 6, 400 it.)"],
    "Mezcla HGB+LGBM+CatBoost+GAM": ["HGB (modelo actual)", "LightGBM (Optuna, 60 intentos)", "CatBoost (depth 6, 400 it.)", "GAM logístico (C=0.3)"],
}
for nm, parts in BLENDS.items():
    oof = np.zeros(len(y)); aucs, aps = [], []
    for tri, vai in FOLDS:
        p = np.mean([rank01(RES[k]["oof"][vai]) for k in parts], axis=0); oof[vai] = p
        aucs.append(roc_auc_score(y[vai], p)); aps.append(average_precision_score(y[vai], p))
    RES[nm] = dict(oof=oof, auc=np.array(aucs), ap=np.array(aps), secs=0)
    print(f"{nm:42s} AUC {np.mean(aucs):.4f} ± {np.std(aucs):.4f} | AP {np.mean(aps):.4f}")

## 4. Comparación en validación cruzada (pliegues idénticos)

In [ ]:
base = "HGB (modelo actual)"
rows = []
for k, r in RES.items():
    d = r["auc"] - RES[base]["auc"]
    rows.append({"modelo": k, "AUC (media)": r["auc"].mean(), "± sd": r["auc"].std(), "PR-AUC": r["ap"].mean(),
                 "Δ AUC vs HGB": d.mean(), "pliegues mejores (de 5)": int((d > 0).sum()), "AUC OOF agregado": roc_auc_score(y, r["oof"])})
cmp = pd.DataFrame(rows).sort_values("AUC (media)", ascending=False).reset_index(drop=True)
display(cmp.round(4))
noise = RES[base]["auc"].std()
print(f"Ruido típico entre pliegues del modelo base (sd de AUC): {noise:.4f}")
fig, ax = plt.subplots(figsize=(8.5, 5.2))
c2 = cmp.iloc[::-1]
ax.barh(c2["modelo"], c2["AUC (media)"], xerr=c2["± sd"], color=[ORANGE if "Mezcla" in m else BLUE for m in c2["modelo"]])
ax.set_xlim(0.60, 0.675); ax.axvline(RES[base]["auc"].mean(), color="k", ls=":", lw=1)
ax.set(xlabel="ROC-AUC en validación cruzada agrupada", title="Todos los modelos con los mismos pliegues")
fig.savefig(FIG / "lgbm_cv_compare.png"); plt.show()

INTERP_CV

## 5. Evaluación final en el conjunto de prueba (una sola vez)

Se fijan los candidatos **antes** de ver la prueba: HGB actual, LightGBM (Optuna), CatBoost y la mezcla de los tres; se reentrena cada uno con todo el entrenamiento. Como referencia adicional se evalúa el **mismo HGB entrenado con la ventana anterior** (todo desde 2013, excluyendo cualquier grupo presente en la prueba para que no haya fuga).

In [ ]:
Xte, yte, gte = feats(te), te.is_bike.values, te.split_group.values
Xte_cb = feats(te, cat_dtype=False)

final = {}
m_hgb = make_hgb_pipeline(CATS, NUMS, SEED).set_params(**{f"model__{k}": v for k, v in hgb_params.items()}).fit(tr[CATS + NUMS], y)
m_lgb = lgb.LGBMClassifier(verbose=-1, random_state=SEED, n_jobs=4, **BEST_LGB).fit(X, y)
m_cb = CatBoostClassifier(iterations=400, learning_rate=0.07, depth=6, l2_leaf_reg=5, verbose=0, random_seed=SEED, thread_count=4, cat_features=CATS).fit(Xcb, y)
p_hgb = m_hgb.predict_proba(te[CATS + NUMS])[:, 1]; p_lgb = m_lgb.predict_proba(Xte)[:, 1]; p_cb = m_cb.predict_proba(Xte_cb)[:, 1]
final["HGB (modelo actual)"] = p_hgb
final["LightGBM (Optuna)"] = p_lgb
final["CatBoost"] = p_cb
final["Mezcla HGB+LGBM+CatBoost"] = np.mean([rank01(p) for p in (p_hgb, p_lgb, p_cb)], axis=0)
te_keys = set(gkey(te)); old_train = mi_old[~gkey(mi_old).isin(te_keys)]   # sin ningún grupo de la prueba
m_old = make_hgb_pipeline(CATS, NUMS, SEED).set_params(**{f"model__{k}": v for k, v in hgb_params.items()}).fit(old_train[CATS + NUMS], old_train.is_bike.values)
final["Modelo anterior (ventana ≥2013, sin grupos de prueba)"] = m_old.predict_proba(te[CATS + NUMS])[:, 1]
print(f"Modelo anterior: entrenado con {len(old_train):,} filas de la ventana anterior, ninguna del grupo de la prueba")
final["Sin información (prevalencia)"] = np.full(len(te), y.mean())

def boot_ci(p, n=300, seed=0):
    rng = np.random.default_rng(seed); uniq = np.unique(gte)
    idx = pd.Series(np.arange(len(gte))).groupby(gte).apply(np.asarray); a, b = [], []
    for _ in range(n):
        ii = np.concatenate([idx[k] for k in rng.choice(uniq, len(uniq))])
        if yte[ii].min() == yte[ii].max(): continue
        a.append(roc_auc_score(yte[ii], p[ii])); b.append(average_precision_score(yte[ii], p[ii]))
    return np.percentile(a, [2.5, 97.5]), np.percentile(b, [2.5, 97.5])

def capture(p, frac, w=None):
    o = np.argsort(-p, kind="stable")[: int(round(frac * len(p)))]
    return (yte[o].sum() / yte.sum()) if w is None else (w[o].sum() / w.sum())

rows = []
for k, p in final.items():
    ci_a, ci_p = boot_ci(p) if "Sin información" not in k else (np.array([.5, .5]), np.array([yte.mean()] * 2))
    dec1 = yte[np.argsort(-p, kind="stable")[: len(p) // 10]].mean() / yte.mean() if "Sin información" not in k else 1.0
    rows.append({"modelo": k, "AUC": roc_auc_score(yte, p), "IC95 AUC": f"[{ci_a[0]:.3f}, {ci_a[1]:.3f}]", "PR-AUC": average_precision_score(yte, p),
                 "IC95 PR-AUC": f"[{ci_p[0]:.3f}, {ci_p[1]:.3f}]", "lift decil 1": dec1, "captura top-30%": capture(p, 0.3) if "Sin información" not in k else 0.3})
TEST = pd.DataFrame(rows); TEST_DISPLAY = TEST.copy()
print(f"Prueba (ventana estable): n = {len(te):,} | prevalencia = {yte.mean():.3f}")
TEST.round(4)

INTERP_TEST

### 5.1 ¿Las diferencias son reales? Prueba pareada por bootstrap de grupos
Los intervalos de cada modelo se traslapan casi siempre; lo relevante es la **diferencia pareada** (mismos remuestreos) frente al HGB re-entrenado.

In [ ]:
def paired_diff(pa, pb, n=500, seed=1):
    rng = np.random.default_rng(seed); uniq = np.unique(gte)
    idx = pd.Series(np.arange(len(gte))).groupby(gte).apply(np.asarray); d_auc, d_ap = [], []
    for _ in range(n):
        ii = np.concatenate([idx[k] for k in rng.choice(uniq, len(uniq))])
        if yte[ii].min() == yte[ii].max(): continue
        d_auc.append(roc_auc_score(yte[ii], pa[ii]) - roc_auc_score(yte[ii], pb[ii]))
        d_ap.append(average_precision_score(yte[ii], pa[ii]) - average_precision_score(yte[ii], pb[ii]))
    f = lambda v: (np.mean(v), *np.percentile(v, [2.5, 97.5]))
    return f(d_auc), f(d_ap)
ref = final["HGB (modelo actual)"]; rows = []
for k in ["LightGBM (Optuna)", "CatBoost", "Mezcla HGB+LGBM+CatBoost"]:
    (a, al, ah), (b, bl, bh) = paired_diff(final[k], ref)
    rows.append({"modelo vs HGB": k, "Δ AUC": a, "IC95 Δ AUC": f"[{al:+.4f}, {ah:+.4f}]", "¿excluye 0?": not (al <= 0 <= ah),
                 "Δ PR-AUC": b, "IC95 Δ PR-AUC": f"[{bl:+.4f}, {bh:+.4f}]"})
pd.DataFrame(rows).round(4)

## 6. Mejor métrica de negocio: capturar **ganancia de bicicletas**, no solo compras

La ganancia por bicicleta difiere por mercado. Ordenar por la probabilidad *p* ignora eso; ordenar por **valor esperado = *p* × ganancia media de una bicicleta en ese país** (calculada solo con entrenamiento) puede capturar más ganancia con los mismos contactos.

In [ ]:
mbike = tr[tr.is_bike == 1].groupby("Country").Profit.mean()
print("Ganancia media por bicicleta (train) por país:"); print(mbike.round(0).to_string())
bike_profit = (te.Profit * te.is_bike).values
mb = te.Country.map(mbike).values
rows = []
for k in ["HGB (modelo actual)", "LightGBM (Optuna)", "Mezcla HGB+LGBM+CatBoost"]:
    p = final[k]
    for crit, s in [("ordenar por p", p), ("ordenar por p × ganancia media/bici país", p * mb)]:
        rows.append({"modelo": k, "criterio": crit, **{f"ganancia-bici capturada top-{int(f*100)}%": capture(s, f, bike_profit) for f in (0.1, 0.2, 0.3)}})
rows.append({"modelo": "Azar", "criterio": "-", **{f"ganancia-bici capturada top-{int(f*100)}%": f for f in (0.1, 0.2, 0.3)}})
VAL = pd.DataFrame(rows); VAL.round(4)

INTERP_VALUE

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
fr = np.linspace(0.02, 0.6, 30)
for k, col in [("HGB (modelo actual)", GRAY), ("LightGBM (Optuna)", BLUE), ("Mezcla HGB+LGBM+CatBoost", ORANGE)]:
    ax[0].plot(fr, [capture(final[k], f) for f in fr], label=k, color=col)
ax[0].plot(fr, fr, "k:", label="Azar"); ax[0].set(xlabel="Fracción contactada", ylabel="Compras de bicicleta capturadas", title="Curva de ganancias (prueba)"); ax[0].legend(fontsize=7)
p = final["Mezcla HGB+LGBM+CatBoost"]
for crit, s, col in [("p", p, BLUE), ("p × valor", p * mb, ORANGE)]:
    ax[1].plot(fr, [capture(s, f, bike_profit) for f in fr], label=f"Orden por {crit}", color=col)
ax[1].plot(fr, fr, "k:", label="Azar"); ax[1].set(xlabel="Fracción contactada", ylabel="Ganancia de bicicletas capturada", title="Capturar valor, no solo compras"); ax[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "lgbm_gains.png"); plt.show()

In [ ]:
# Guardar resultados para reutilizarlos en el reporte si se decide adoptarlos
out = {"cv": cmp.round(5).to_dict(orient="records"), "test": TEST.round(5).to_dict(orient="records"), "valor": VAL.round(5).to_dict(orient="records"),
       "lgbm_best_params": BEST_LGB, "n_train": int(len(tr)), "n_test": int(len(te)), "prevalence_test": float(yte.mean())}
(ROOT / "docs").mkdir(exist_ok=True)
json.dump(out, open(ROOT / "docs" / "lgbm_experiment_results.json", "w", encoding="utf-8"), indent=1, ensure_ascii=False, default=str)
pd.to_pickle({"lgbm": m_lgb, "catboost": m_cb, "hgb_strict": m_hgb}, DATA / "06_models" / "experiment_models.pkl")
print("Guardado: docs/lgbm_experiment_results.json y data/06_models/experiment_models.pkl")

CONCLUSIONES

## Referencias

- Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining*, 2623–2631.
- Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems, 30*.
- Prokhorenkova, L., Gusev, G., Vorobev, A., Dorogush, A. V., & Gulin, A. (2018). CatBoost: Unbiased boosting with categorical features. *Advances in Neural Information Processing Systems, 31*.
- Fisher, W. D. (1958). On grouping for maximum homogeneity. *Journal of the American Statistical Association, 53*(284), 789–798.
- Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall.
- Kaufman, S., Rosset, S., Perlich, C., & Stitelman, O. (2012). Leakage in data mining: Formulation, detection, and avoidance. *ACM Transactions on Knowledge Discovery from Data, 6*(4), 15.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.
- Documentación: LightGBM (<https://lightgbm.readthedocs.io>), CatBoost (<https://catboost.ai/docs>), Optuna (<https://optuna.readthedocs.io>).

> Los datos bibliográficos deben verificarse contra la fuente original antes de entregar (no se pudieron comprobar en línea).